In [1]:
!pip install albumentations opencv-python


  Using cached numpy-2.2.6-cp310-cp310-win_amd64.whl.metadata (60 kB)
  Using cached scipy-1.15.3-cp310-cp310-win_amd64.whl.metadata (60 kB)
  Using cached opencv_python_headless-4.13.0.90-cp37-abi3-win_amd64.whl.metadata (20 kB)
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
   ---------------------------------------- 0.0/40.2 MB ? eta -:--:--
   ---------------------------------------- 0.3/40.2 MB ? eta -:--:--
   ---------------------------------------- 0.3/40.2 MB ? eta -:--:--
    --------------------------------------- 0.8/40.2 MB 1.3 MB/s eta 0:00:30
   - -------------------------------------- 1.0/40.2 MB 1.4 MB/s eta 0:00:28
   - -------------------------------------- 1.6/40.2 MB 1.6 MB/s eta 0:00:24
   -- ------------------------------------- 2.1/40.2 MB 1.8 MB/s eta 0:00:22
   -- ------------------------------------- 2.6/40.2 MB 2.0 MB/s eta 0:00:20
   --- ------------------------------------ 3.1/40.2 MB 2.0 MB/s eta 0:00:19
   --- --------------------

In [2]:
import os
import cv2
import albumentations as A


In [4]:
import albumentations as A

transform = A.Compose(
    [
        A.HorizontalFlip(p=0.5),
        A.RandomBrightnessContrast(p=0.4),
        A.Rotate(limit=10, p=0.4),
        A.Blur(blur_limit=3, p=0.2),
        A.Affine(scale=(0.9, 1.1), p=0.4)  # replaces A.Scale
    ],
    bbox_params=A.BboxParams(
        format='yolo',
        label_fields=['class_labels']
    )
)


In [5]:
base = r"D:\TEAM IDEATORS\1_objective\animal_dataset"
classes = ["dog", "cow", "elephant"]

for cls in classes:
    img_dir = os.path.join(base, "images", "train", cls)
    lbl_dir = os.path.join(base, "labels", "train", cls)

    images = [f for f in os.listdir(img_dir) if f.endswith(".jpg")]

    for img_name in images:
        img_path = os.path.join(img_dir, img_name)
        lbl_path = os.path.join(lbl_dir, img_name.replace(".jpg", ".txt"))

        image = cv2.imread(img_path)
        h, w, _ = image.shape

        boxes = []
        class_labels = []

        with open(lbl_path, "r") as f:
            for line in f:
                cls_id, x, y, bw, bh = map(float, line.split())
                boxes.append([x, y, bw, bh])
                class_labels.append(int(cls_id))

        augmented = transform(
            image=image,
            bboxes=boxes,
            class_labels=class_labels
        )

        aug_img_name = "aug_" + img_name
        aug_lbl_name = aug_img_name.replace(".jpg", ".txt")

        cv2.imwrite(
            os.path.join(img_dir, aug_img_name),
            augmented["image"]
        )

        with open(os.path.join(lbl_dir, aug_lbl_name), "w") as f:
            for lbl, box in zip(class_labels, augmented["bboxes"]):
                f.write(f"{lbl} {' '.join(map(str, box))}\n")

print("✅ Augmentation completed successfully")


✅ Augmentation completed successfully
